# 13 — Object-level LeJEPA with masks

**Local result type:** exploratory transfer study. This notebook does not reproduce the paper's COCO benchmark.

## Learning goals

- Compare image-level and mask-pooled self-supervised objectives.
- Inspect how a human-provided foreground trimap changes the training signal.
- Measure pet-breed classification with frozen features on the official test split.
- Keep this local result separate from the paper's reported findings.

In [ ]:
import hashlib
import os
import platform
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from scipy.stats import t as student_t
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

repo_root = Path.cwd()
if not (repo_root / "src" / "jepa_tutorial").is_dir():
    raise RuntimeError("Run this notebook from the repository root.")

sys.path.insert(0, str(repo_root / "src"))
from jepa_tutorial.storage import DATA_DIR
from jepa_tutorial.paper_losses import epps_pulley_sigreg_loss

torch.set_num_threads(min(8, os.cpu_count() or 1))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "| TorchVision:", torchvision.__version__)
print("NumPy:", np.__version__, "| SciPy:", scipy.__version__, "| scikit-learn:", sklearn.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Source paper and mask data

The current paper version is v2 of [Mask-supervised Object-centric Representation Learning with LeJEPA](https://arxiv.org/abs/2607.02404). Its main COCO experiment uses human ground-truth instance masks. SAM proposals and Felzenszwalb–Huttenlocher superpixels are separate mask-source ablations. The method aligns per-object semantic representations with LeJEPA SIGReg, separates instances in a second feature space, and uses copy-paste views to change object context.

The paper reports ADE20k object-classification balanced accuracy of 0.555 for its semantic object representation and 0.426 for image-level LeJEPA. It reports DAVIS tracking J&F of 0.683 and 0.626. These are paper values, not results from this notebook.

This study uses the [Oxford-IIIT Pet dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/) through the [TorchVision dataset interface](https://docs.pytorch.org/vision/0.28/generated/torchvision.datasets.OxfordIIITPet.html). The dataset page lists 37 categories and pixel-level foreground/background trimaps. The download is about 800 MB and is under CC BY-SA 4.0; image copyright remains with the original owners. The trimap provides pet, background, and outline regions. This notebook treats pet and outline as foreground. It excludes trainval images with no foreground trimap before balanced sampling. It retains the full test split and uses global-average pooling for both methods when a test trimap has no foreground; the notebook reports this fallback count.

In [ ]:
DATA_ROOT = DATA_DIR
DATA_SEED = 20261014
TRAIN_PER_CLASS = 16
IMAGE_SIZE = 96
BATCH_SIZE = 64
NUM_EPOCHS = 5
TRAIN_SEEDS = [7, 17, 27, 37, 47]
METHODS = ["image_level", "object_level"]
POOLING_MODES = ["global_average", "pet_mask_with_global_fallback"]

dataset_load_started = time.perf_counter()
try:
    trainval_dataset = datasets.OxfordIIITPet(
        root=str(DATA_ROOT),
        split="trainval",
        target_types=("category", "segmentation"),
        download=True,
    )
    test_dataset = datasets.OxfordIIITPet(
        root=str(DATA_ROOT),
        split="test",
        target_types=("category", "segmentation"),
        download=True,
    )
except Exception as exc:
    dataset_load_seconds = time.perf_counter() - dataset_load_started
    dataset_load_error = f"{type(exc).__name__}: {exc}"
    DATASET_LOAD_FAILURES = [
        {
            "method": method,
            "seed": seed,
            "status": "failed",
            "stage": "dataset_load",
            "device": str(DEVICE),
            "dataset_load_seconds": round(dataset_load_seconds, 1),
            "error": dataset_load_error,
        }
        for seed in TRAIN_SEEDS
        for method in METHODS
    ]
    print("DATASET LOAD FAILED; comparison remains incomplete.")
    print("No model training or probe ran.")
    print("Error:", dataset_load_error)
    print("Failed run records:")
    for failure in DATASET_LOAD_FAILURES:
        print(failure)
    raise
print("Dataset load seconds:", round(time.perf_counter() - dataset_load_started, 1))

if len(trainval_dataset.classes) != 37:
    raise ValueError(f"Expected 37 categories, got {len(trainval_dataset.classes)}")

# Read labels and mask validity through the official dataset interface.
# Breed labels select the balanced subset; labels are not passed to the loss.
label_scan_started = time.perf_counter()
trainval_labels_list = []
trainval_mask_valid_list = []
for index in range(len(trainval_dataset)):
    _, (label, trimap) = trainval_dataset[index]
    resized_trimap = TF.resize(
        trimap,
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=InterpolationMode.NEAREST,
    )
    foreground = np.isin(np.asarray(resized_trimap, dtype=np.uint8), (1, 3))
    trainval_labels_list.append(label)
    trainval_mask_valid_list.append(bool(foreground.any()))
trainval_labels = np.asarray(trainval_labels_list, dtype=np.int64)
trainval_mask_valid = np.asarray(trainval_mask_valid_list, dtype=bool)
trainval_empty_mask_count = int((~trainval_mask_valid).sum())
test_empty_mask_count = 0
for index in range(len(test_dataset)):
    _, (_, trimap) = test_dataset[index]
    resized_trimap = TF.resize(
        trimap,
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=InterpolationMode.NEAREST,
    )
    if not np.isin(np.asarray(resized_trimap, dtype=np.uint8), (1, 3)).any():
        test_empty_mask_count += 1
label_scan_seconds = time.perf_counter() - label_scan_started
rng = np.random.default_rng(DATA_SEED)
selected = []
class_counts = {}
for category in range(37):
    category_indices = np.flatnonzero(
        (trainval_labels == category) & trainval_mask_valid
    )
    if category_indices.size < TRAIN_PER_CLASS:
        raise ValueError(
            f"Category {category} has only {category_indices.size} trainval images with foreground masks"
        )
    chosen = np.sort(rng.choice(category_indices, size=TRAIN_PER_CLASS, replace=False))
    selected.extend(chosen.tolist())
    class_counts[category] = int(chosen.size)

TRAIN_INDICES = np.asarray(sorted(selected), dtype=np.int64)
SUBSET_SHA256 = hashlib.sha256(TRAIN_INDICES.tobytes()).hexdigest()
print("Trainval images:", len(trainval_dataset))
print("One-time label and mask scan seconds:", round(label_scan_seconds, 1))
print("Trainval examples with no foreground mask excluded:", trainval_empty_mask_count)
print("Official test images:", len(test_dataset))
print("Test examples using global-pooling fallback:", test_empty_mask_count)
print("Categories:", len(trainval_dataset.classes))
print("Balanced unlabeled training subset:", len(TRAIN_INDICES), "(16 per category with valid masks)")
print("Subset index SHA-256:", SUBSET_SHA256)

The self-supervised training loss receives two image views and, for the object-level arm, the aligned foreground masks. It does not receive breed labels. The fixed trainval subset has 16 images per category. The full official test split is used for the final probe score.

In [ ]:
example_image, (example_label, example_trimap) = trainval_dataset[int(TRAIN_INDICES[0])]
example_trimap_array = np.asarray(example_trimap, dtype=np.uint8)
example_foreground = np.isin(example_trimap_array, (1, 3))

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(example_image)
axes[0].set_title("Input image")
axes[1].imshow(example_trimap_array, cmap="viridis", vmin=1, vmax=3)
axes[1].set_title("Human trimap")
axes[2].imshow(example_image)
axes[2].imshow(example_foreground, alpha=0.4, cmap="Reds")
axes[2].set_title("Pet + outline foreground")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Example category:", trainval_dataset.classes[int(example_label)])
print("Trimap values:", np.unique(example_trimap_array).tolist())

## 2. Paired views and mask pooling

Both methods use the same resize, horizontal-flip, and color-jitter distribution. The object-level arm pools encoder features only over the pet and outline pixels. The image-level arm pools over the full feature map. The trimap uses nearest-neighbour resizing so it stays discrete.

In [ ]:
IMAGE_MEAN = (0.485, 0.456, 0.406)
IMAGE_STD = (0.229, 0.224, 0.225)

def make_view(image, trimap):
    image = TF.resize(
        image.convert("RGB"),
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=InterpolationMode.BILINEAR,
    )
    trimap = TF.resize(
        trimap,
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=InterpolationMode.NEAREST,
    )

    if random.random() < 0.5:
        image = TF.hflip(image)
        trimap = TF.hflip(trimap)

    if random.random() < 0.8:
        image = TF.adjust_brightness(image, random.uniform(0.8, 1.2))
        image = TF.adjust_contrast(image, random.uniform(0.8, 1.2))
        image = TF.adjust_saturation(image, random.uniform(0.8, 1.2))

    trimap_array = np.asarray(trimap, dtype=np.uint8)
    foreground = np.isin(trimap_array, (1, 3)).astype(np.float32)
    if foreground.sum() == 0:
        raise ValueError("The resized Oxford-IIIT Pet trimap has no foreground pixels")

    image_tensor = TF.normalize(TF.to_tensor(image), IMAGE_MEAN, IMAGE_STD)
    mask_tensor = torch.from_numpy(foreground.copy()).unsqueeze(0)
    return image_tensor, mask_tensor

class OxfordTwoViewDataset(Dataset):
    def __init__(self, base_dataset, indices):
        self.base_dataset = base_dataset
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        image, (_, trimap) = self.base_dataset[int(self.indices[item])]
        views = [make_view(image, trimap) for _ in range(2)]
        return (
            torch.stack([view[0] for view in views]),
            torch.stack([view[1] for view in views]),
        )

class OxfordEvalDataset(Dataset):
    def __init__(self, base_dataset, indices):
        self.base_dataset = base_dataset
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        image, (label, trimap) = self.base_dataset[int(self.indices[item])]
        image = TF.resize(
            image.convert("RGB"),
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=InterpolationMode.BILINEAR,
        )
        trimap = TF.resize(
            trimap,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=InterpolationMode.NEAREST,
        )
        foreground = np.isin(np.asarray(trimap, dtype=np.uint8), (1, 3)).astype(np.float32)
        mask_valid = bool(foreground.any())
        image_tensor = TF.normalize(TF.to_tensor(image), IMAGE_MEAN, IMAGE_STD)
        mask_tensor = torch.from_numpy(foreground.copy()).unsqueeze(0)
        return image_tensor, mask_tensor, int(label), mask_valid

def pool_features(feature_map, mask=None):
    if mask is None:
        return feature_map.mean(dim=(2, 3))
    resized_mask = F.interpolate(mask.float(), size=feature_map.shape[-2:], mode="nearest")
    weight_sum = resized_mask.sum(dim=(2, 3)).clamp_min(1.0)
    return (feature_map * resized_mask).sum(dim=(2, 3)) / weight_sum

print("Views per image: 2")
print("Input size:", IMAGE_SIZE, "x", IMAGE_SIZE)
print("Training examples per class:", TRAIN_PER_CLASS)

## 3. Small encoder and objective

The encoder is a small CNN trained from scratch. Both arms use the same model, projection head, optimizer, two views, and number of epochs. The object-level arm applies alignment and SIGReg to mask-pooled features. The image-level arm applies them to full-image pooled features.

This is a reduced objective study. It does not implement the paper's separate instance-separation projection or its copy-paste augmentation.

In [ ]:
class SmallEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

    def forward(self, image):
        return self.features(image)

class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(256, 512),
            nn.GELU(),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Linear(256, 64),
        )

    def forward(self, features):
        return self.layers(features)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train_method(method, seed):
    set_seed(seed)
    dataset = OxfordTwoViewDataset(trainval_dataset, TRAIN_INDICES)
    loader_generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=loader_generator,
        num_workers=0,
        drop_last=False,
    )

    encoder = SmallEncoder().to(DEVICE)
    projector = ProjectionHead().to(DEVICE)
    parameters = list(encoder.parameters()) + list(projector.parameters())
    optimizer = torch.optim.AdamW(parameters, lr=5e-4, weight_decay=5e-4)
    epoch_losses = []
    started = time.perf_counter()

    for epoch in range(NUM_EPOCHS):
        encoder.train()
        projector.train()
        losses = []
        for batch_index, (views, masks) in enumerate(loader):
            views = views.to(DEVICE)
            masks = masks.to(DEVICE)
            projected_views = []
            for view_index in range(2):
                feature_map = encoder(views[:, view_index])
                mask = masks[:, view_index] if method == "object_level" else None
                pooled = pool_features(feature_map, mask)
                projected_views.append(projector(pooled))

            projected = torch.stack(projected_views, dim=0)
            center = projected.mean(dim=0, keepdim=True)
            alignment = (projected - center).square().mean()
            sigreg_terms = []
            for view_index in range(2):
                sig_generator = torch.Generator(device=DEVICE)
                sig_generator.manual_seed(seed * 100000 + epoch * 1000 + batch_index * 2 + view_index)
                sigreg_terms.append(
                    epps_pulley_sigreg_loss(
                        projected_views[view_index],
                        num_projections=128,
                        num_points=17,
                        generator=sig_generator,
                    )
                )
            sigreg = torch.stack(sigreg_terms).mean()
            loss = alignment + 0.05 * sigreg

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(parameters, max_norm=1.0)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        epoch_losses.append(float(np.mean(losses)))

    elapsed = time.perf_counter() - started
    return encoder.eval(), {
        "training_seconds": elapsed,
        "epoch_losses": epoch_losses,
        "parameter_count": sum(parameter.numel() for parameter in encoder.parameters()),
    }

print("Training settings:", {
    "batch_size": BATCH_SIZE,
    "epochs": NUM_EPOCHS,
    "seeds": TRAIN_SEEDS,
    "SIGReg_projections": 128,
    "SIGReg_points": 17,
    "device": str(DEVICE),
})

## 4. Frozen pet-breed probe

The probe uses the last encoder feature map. For each trained model, it tests two pooling rules:

- Global average pooling over the full image.
- Foreground-mask pooling over the pet and outline pixels. If a test trimap has no foreground pixels, use global-average pooling for that example for both methods.

This gives a small 2-by-2 comparison: training objective (image or object) and feature pooling (global or mask with a shared fallback). The primary local comparison uses the same mask-pooling rule for both training arms. All probes use the same trainval images and full official test split. The notebook reports how many test examples use the global-pooling fallback.

In [ ]:
def extract_feature_tables(encoder, eval_dataset, batch_size=128):
    loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    global_parts = []
    mask_parts = []
    label_parts = []
    encoder.eval()
    with torch.inference_mode():
        for images, masks, labels, mask_valid in loader:
            feature_map = encoder(images.to(DEVICE))
            global_features = pool_features(feature_map)
            mask_features = pool_features(feature_map, masks.to(DEVICE))
            valid = mask_valid.to(DEVICE).unsqueeze(1)
            mask_features = torch.where(valid, mask_features, global_features)
            global_parts.append(global_features.cpu().numpy())
            mask_parts.append(mask_features.cpu().numpy())
            label_parts.append(labels.numpy())
    return (
        np.concatenate(global_parts),
        np.concatenate(mask_parts),
        np.concatenate(label_parts),
    )

def probe_score(train_features, train_labels, test_features, test_labels, seed):
    probe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            C=1.0,
            random_state=seed,
        ),
    )
    probe.fit(train_features, train_labels)
    predictions = probe.predict(test_features)
    return float(balanced_accuracy_score(test_labels, predictions))

train_eval_dataset = OxfordEvalDataset(trainval_dataset, TRAIN_INDICES)
test_eval_dataset = OxfordEvalDataset(test_dataset, np.arange(len(test_dataset)))
probe_train_labels = trainval_labels[TRAIN_INDICES]

In [ ]:
RUNS = []
RUN_STARTED = time.perf_counter()

for seed in TRAIN_SEEDS:
    for method in METHODS:
        try:
            if DEVICE.type == "cuda":
                torch.cuda.reset_peak_memory_stats()
            encoder, training_record = train_method(method, seed)
            feature_started = time.perf_counter()
            train_global, train_mask, train_labels = extract_feature_tables(
                encoder, train_eval_dataset
            )
            test_global, test_mask, test_labels = extract_feature_tables(
                encoder, test_eval_dataset
            )
            feature_seconds = time.perf_counter() - feature_started
            peak_cuda_mb = (
                float(torch.cuda.max_memory_allocated() / (1024**2))
                if DEVICE.type == "cuda" else None
            )

            scores = {
                "global_average": probe_score(
                    train_global, train_labels, test_global, test_labels, seed
                ),
                "pet_mask_with_global_fallback": probe_score(
                    train_mask, train_labels, test_mask, test_labels, seed
                ),
            }
            RUNS.append({
                "method": method,
                "seed": seed,
                "status": "completed",
                "training_seconds": training_record["training_seconds"],
                "feature_seconds": feature_seconds,
                "peak_cuda_mb": peak_cuda_mb,
                "device": str(DEVICE),
                "feature_parameter_count": training_record["parameter_count"],
                "epoch_losses": training_record["epoch_losses"],
                "scores": scores,
            })
            del encoder
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            print(
                method,
                "seed", seed,
                "global", f"{scores['global_average']:.4f}",
                "pet-mask", f"{scores['pet_mask_with_global_fallback']:.4f}",
                "train_s", f"{training_record['training_seconds']:.1f}",
                "features_s", f"{feature_seconds:.1f}",
            )
        except Exception as exc:
            RUNS.append({
                "method": method,
                "seed": seed,
                "status": "failed",
                "error": f"{type(exc).__name__}: {exc}",
            })
            print("FAILED:", method, "seed", seed, RUNS[-1]["error"])

TOTAL_STUDY_SECONDS = time.perf_counter() - RUN_STARTED
print("Study seconds:", round(TOTAL_STUDY_SECONDS, 1))
print("Completed model runs:", sum(run["status"] == "completed" for run in RUNS), "of", len(RUNS))
print("Trimap foreground rule: values 1 and 3")

In [ ]:
def summarize(values):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        return {"n": 0, "mean": np.nan, "median": np.nan, "low": np.nan, "high": np.nan}
    mean = float(values.mean())
    median = float(np.median(values))
    if values.size < 2:
        return {"n": int(values.size), "mean": mean, "median": median, "low": np.nan, "high": np.nan}
    sem = float(values.std(ddof=1) / np.sqrt(values.size))
    half_width = float(student_t.ppf(0.975, df=values.size - 1) * sem)
    return {
        "n": int(values.size),
        "mean": mean,
        "median": median,
        "low": mean - half_width,
        "high": mean + half_width,
    }

SUMMARY = {}
print("method        pooling          n   mean     median   95% CI")
for method in METHODS:
    method_runs = [run for run in RUNS if run["method"] == method and run["status"] == "completed"]
    for pooling in POOLING_MODES:
        values = [run["scores"][pooling] for run in method_runs]
        summary = summarize(values)
        SUMMARY[(method, pooling)] = summary
        print(
            f"{method:13s} {pooling:15s} {summary['n']:2d} "
            f"{summary['mean']:.4f}  {summary['median']:.4f}  "
            f"[{summary['low']:.4f}, {summary['high']:.4f}]"
        )

paired_mask = []
for seed in TRAIN_SEEDS:
    image_run = next(
        (run for run in RUNS if run["method"] == "image_level" and run["seed"] == seed and run["status"] == "completed"),
        None,
    )
    object_run = next(
        (run for run in RUNS if run["method"] == "object_level" and run["seed"] == seed and run["status"] == "completed"),
        None,
    )
    if image_run and object_run:
        paired_mask.append(object_run["scores"]["pet_mask_with_global_fallback"] - image_run["scores"]["pet_mask_with_global_fallback"])

paired_summary = summarize(paired_mask)
print("Matched object-minus-image difference on pet-mask/fallback features:")
print({
    "per_seed": [round(value, 4) for value in paired_mask],
    "mean": paired_summary["mean"],
    "median": paired_summary["median"],
    "95%_CI": [paired_summary["low"], paired_summary["high"]],
    "n": paired_summary["n"],
})

FIVE_SEED_COMPARISON_COMPLETE = all(
    SUMMARY[(method, pooling)]["n"] == len(TRAIN_SEEDS)
    for method in METHODS
    for pooling in POOLING_MODES
)
print("Five-seed comparison complete:", FIVE_SEED_COMPARISON_COMPLETE)
if not FIVE_SEED_COMPARISON_COMPLETE:
    print("Incomplete run: keep these values provisional; do not report a completed comparison.")


In [ ]:
labels = [
    ("Image objective", "Global pool", "image_level", "global_average"),
    ("Image objective", "Pet-mask pool", "image_level", "pet_mask_with_global_fallback"),
    ("Object objective", "Global pool", "object_level", "global_average"),
    ("Object objective", "Pet-mask pool", "object_level", "pet_mask_with_global_fallback"),
]
fig, ax = plt.subplots(figsize=(9, 4.5))
positions = np.arange(len(labels))
for position, (method_label, pooling_label, method, pooling) in zip(positions, labels):
    runs = [run for run in RUNS if run["method"] == method and run["status"] == "completed"]
    values = np.asarray([run["scores"][pooling] for run in runs], dtype=float)
    summary = summarize(values)
    if values.size:
        ax.bar(position, summary["mean"], color="#4c78a8" if method == "image_level" else "#f58518")
    else:
        ax.text(position, 0.02, "n=0", ha="center", va="bottom")
    if values.size > 1:
        ax.errorbar(
            position,
            summary["mean"],
            yerr=[[summary["mean"] - summary["low"]], [summary["high"] - summary["mean"]]],
            color="black",
            capsize=4,
        )
    if values.size:
        ax.scatter(
            np.full(values.size, position) + np.linspace(-0.08, 0.08, values.size),
            values,
            color="black",
            s=20,
            zorder=3,
        )
ax.set_xticks(positions, [f"{a}\n{b}" for a, b, _, _ in labels])
ax.set_ylabel("Balanced test accuracy")
ax.set_ylim(0, 1)
ax.set_title(
    "Frozen Oxford-IIIT Pet breed probe"
    + ("" if FIVE_SEED_COMPARISON_COMPLETE else " — incomplete run")
)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## What this evidence supports

This local study tests a reduced object-pooled objective on Oxford-IIIT Pet. It uses 592 trainval images with nonempty resized masks, a small CNN trained from scratch, 96-pixel views, five seeds, five epochs, and a balanced linear probe on the full official test split. Test images without foreground pixels use global-average pooling for both methods; the fallback count is printed with the data summary.

It changes the paper's dataset, model, resolution, training duration, SIGReg projection count, objective, and downstream task. It omits the instance-separation loss and copy-paste augmentation. It therefore does not reproduce the paper's COCO, ADE20k, DAVIS, NAVI, or ImageNet results and cannot validate the paper's benchmark findings.

The paper's v2 reproducibility statement says its object-centric training and SAM-mask code will be released. The paper source review on 2026-10-14 did not find a linked first-party implementation. The project keeps the exact paper benchmark open in the reproduction matrix.